# 🫀 MLOps Assignment 01: Exploratory Data Analysis & Model Development
**Course**: AIML ZG523 (MLOps) — BITS Pilani WILP  
**Student**: Kiran Kumar (`2025AE05377`)  
**Dataset**: UCI Heart Disease (Cleveland Database)  

This notebook walks through the data ingestion, statistical exploratory analysis, leak-free pipeline preprocessing, multi-model training (Logistic Regression vs. Random Forest), and MLflow experiment tracking.

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

# Robust root path detection (works whether executed from notebooks/ or project root)
PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from data_prep import prepare_data, FEATURE_COLS, NUMERICAL_COLS, CATEGORICAL_COLS
print('Environment initialized successfully.')
print(f'Project root detected at: {PROJECT_ROOT}')

## 1. Data Ingestion & Schema Inspection

In [2]:
data_path = PROJECT_ROOT / 'data' / 'processed' / 'heart_cleaned.csv'
df = pd.read_csv(data_path)
print(f'Dataset Shape: {df.shape}')
df.head()

In [3]:
df.info()
df.describe().T

## 2. Target Class Distribution (Class Balance)

In [4]:
plt.figure(figsize=(6, 4))
counts = df['target'].value_counts()
plt.bar(['Absence (0)', 'Presence (1)'], counts.values, color=['#2b5c8f', '#d95f02'], width=0.5)
plt.title('Target Variable Distribution (Cleveland Clinic)')
plt.ylabel('Patient Count')
for i, v in enumerate(counts.values):
    plt.text(i, v + 3, f'{v} ({v/len(df)*100:.1f}%)', ha='center', fontweight='bold')
plt.ylim(0, max(counts.values) + 25)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

## 3. Correlation Heatmap Analysis

In [5]:
plt.figure(figsize=(10, 8))
corr = df.corr()
plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(shrink=0.8)
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha='right')
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title('Pearson Correlation Heatmap of Clinical Variables')
plt.tight_layout()
plt.show()

## 4. Key Physiological Risk Indicators (Distributions & Boxplots)

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
key_vars = [('thalach', 'Max Heart Rate Achieved (bpm)'), ('oldpeak', 'ST Depression (mm)'), ('chol', 'Serum Cholesterol (mg/dl)')]
for idx, (col, title) in enumerate(key_vars):
    ax = axes[idx]
    data_groups = [df[df['target']==0][col].dropna(), df[df['target']==1][col].dropna()]
    ax.boxplot(data_groups, tick_labels=['Absence', 'Presence'], patch_artist=True)
    ax.set_title(title, fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 5. Model Inspection & Inference Verification
Load champion serialized pipeline (`best_model.joblib`) and test inference.

In [7]:
model_path = PROJECT_ROOT / 'artifacts' / 'best_model.joblib'
pipeline = joblib.load(model_path)
print('Loaded Pipeline Steps:', [step[0] for step in pipeline.steps])

sample_patient = pd.DataFrame([{
    'age': 60, 'sex': 1, 'cp': 4, 'trestbps': 145, 'chol': 282, 'fbs': 0,
    'restecg': 2, 'thalach': 142, 'exang': 1, 'oldpeak': 2.8, 'slope': 2, 'ca': 2.0, 'thal': 7.0
}])

pred = pipeline.predict(sample_patient)[0]
proba = pipeline.predict_proba(sample_patient)[0][1]
print(f'Prediction: {pred} (Heart Disease Present)' if pred == 1 else f'Prediction: {pred} (Heart Disease Absent)')
print(f'Calculated Probability: {proba:.4f}')